<a href="https://colab.research.google.com/github/akshithaarakshi-lab/generative-ai-and-ai-agents-development/blob/main/Janmitra_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# JANMITRA AI
# Karnataka Scheme Eligibility Assistant
# Google Colab + Gradio
#
# IMPORTANT:
# Yuva Nidhi rules below are a HISTORICAL reference snapshot.
# They are not a confirmation of eligibility under current rules.
# ============================================================

!pip -q install -U gradio

import gradio as gr
from datetime import datetime

# ------------------------------------------------------------
# OFFICIAL / GOVERNMENT SOURCES
# ------------------------------------------------------------

SEVA_ONE = "https://sevaone.karnataka.gov.in/"
ELIGIBILITY_URL = (
    "https://sevaone.karnataka.gov.in/services/"
    "knowYourEligibility.html"
)
MYSCHEME_KARNATAKA = (
    "https://www.myscheme.gov.in/search/state/Karnataka"
)

# Government of Karnataka research report describing Yuva Nidhi
# historical criteria:
YUVANIDHI_REPORT = (
    "https://fpibengaluru.karnataka.gov.in/uploads/"
    "media_to_upload1762775086.pdf"
)

# ------------------------------------------------------------
# CONDITION EVALUATION
# Each condition returns MATCHED, NOT MATCHED, or UNKNOWN.
# ------------------------------------------------------------

def evaluate_yes_no(value, expected=True):
    if value == "Not sure":
        return "UNKNOWN"

    if expected:
        return "MATCHED" if value == "Yes" else "NOT MATCHED"

    return "MATCHED" if value == "No" else "NOT MATCHED"


def evaluate_yuva_nidhi(
    state_connection,
    qualification,
    completion_year,
    unemployed_days,
    employment_status,
    studying_higher,
    paid_apprenticeship,
    self_employed_with_loan,
    registered,
    declaration
):
    results = []

    def add(condition, status, explanation):
        results.append({
            "condition": condition,
            "status": status,
            "explanation": explanation
        })

    # 1. Karnataka connection
    if state_connection == "Not sure":
        add(
            "Karnataka residency / state connection",
            "UNKNOWN",
            "The applicant's state connection has not been confirmed."
        )
    elif state_connection == "Yes":
        add(
            "Karnataka residency / state connection",
            "MATCHED",
            "The applicant reports meeting the six-year Karnataka "
            "connection described in the historical reference. "
            "Official verification is still required."
        )
    else:
        add(
            "Karnataka residency / state connection",
            "NOT MATCHED",
            "The applicant reports not meeting the historical "
            "six-year Karnataka connection criterion."
        )

    # 2. Qualification
    if qualification == "Not sure":
        add(
            "Graduation or diploma qualification",
            "UNKNOWN",
            "The qualification has not been confirmed."
        )
    elif qualification in ["Degree", "Diploma"]:
        add(
            "Graduation or diploma qualification",
            "MATCHED",
            "The qualification type matches the historical reference."
        )
    else:
        add(
            "Graduation or diploma qualification",
            "NOT MATCHED",
            "The reported qualification does not match the "
            "graduation/diploma condition in this reference snapshot."
        )

    # 3. Historical cohort
    if completion_year == "Not sure":
        add(
            "Course completion cohort",
            "UNKNOWN",
            "The completion academic year is unknown."
        )
    elif completion_year == "2022-23":
        add(
            "Course completion cohort",
            "MATCHED",
            "The year matches the 2022-23 cohort described in the "
            "historical government report. Current cohort rules "
            "may differ."
        )
    else:
        add(
            "Course completion cohort",
            "NOT MATCHED",
            "The year does not match the 2022-23 cohort in this "
            "historical snapshot. This is not a determination under "
            "current scheme rules."
        )

    # 4. Unemployment duration
    if unemployed_days is None:
        add(
            "At least 180 days unemployed after course completion",
            "UNKNOWN",
            "The unemployment duration was not supplied."
        )
    elif unemployed_days >= 180:
        add(
            "At least 180 days unemployed after course completion",
            "MATCHED",
            "The reported duration is 180 days or more. The official "
            "authority must verify the actual period and status."
        )
    else:
        add(
            "At least 180 days unemployed after course completion",
            "NOT MATCHED",
            "The reported duration is below 180 days under the "
            "historical reference condition."
        )

    # 5. Employment status
    if employment_status == "Not sure":
        add(
            "Current employment status",
            "UNKNOWN",
            "Employment status needs confirmation."
        )
    elif employment_status == "Unemployed":
        add(
            "Current employment status",
            "MATCHED",
            "The applicant reports being unemployed."
        )
    else:
        add(
            "Current employment status",
            "NOT MATCHED",
            "The applicant reports being employed or self-employed. "
            "This conflicts with the historical unemployed-applicant "
            "condition; current exceptions must be checked officially."
        )

    # 6. Higher education exclusion
    status = evaluate_yes_no(studying_higher, expected=False)
    add(
        "Not enrolled in higher education",
        status,
        (
            "The applicant reports not pursuing higher education."
            if status == "MATCHED" else
            "The applicant reports pursuing higher education."
            if status == "NOT MATCHED" else
            "Higher education status is unknown."
        )
    )

    # 7. Paid apprenticeship exclusion
    status = evaluate_yes_no(paid_apprenticeship, expected=False)
    add(
        "No paid apprenticeship that conflicts with the scheme rules",
        status,
        (
            "The applicant reports no paid apprenticeship."
            if status == "MATCHED" else
            "The applicant reports a paid apprenticeship. Verify "
            "the current scheme's exclusion rules."
            if status == "NOT MATCHED" else
            "Apprenticeship status is unknown."
        )
    )

    # 8. Self-employment with qualifying loan exclusion
    status = evaluate_yes_no(self_employed_with_loan, expected=False)
    add(
        "Self-employment with a relevant bank/government loan",
        status,
        (
            "The applicant reports no such self-employment loan."
            if status == "MATCHED" else
            "The applicant reports self-employment with a relevant "
            "loan, which conflicts with the historical exclusion."
            if status == "NOT MATCHED" else
            "The applicant's loan and self-employment situation "
            "is unknown."
        )
    )

    # 9. Registration
    status = evaluate_yes_no(registered, expected=True)
    add(
        "Registration on the prescribed government portal",
        status,
        (
            "The applicant reports having registered."
            if status == "MATCHED" else
            "The applicant reports not being registered."
            if status == "NOT MATCHED" else
            "Registration has not been confirmed."
        )
    )

    # 10. Declaration
    status = evaluate_yes_no(declaration, expected=True)
    add(
        "Required unemployment declaration",
        status,
        (
            "The applicant reports submitting the declaration."
            if status == "MATCHED" else
            "The applicant reports not submitting the declaration."
            if status == "NOT MATCHED" else
            "Declaration status is unknown."
        )
    )

    return results


# ------------------------------------------------------------
# CREATE CITIZEN-FRIENDLY REPORT
# ------------------------------------------------------------

def build_report(
    scheme,
    state_connection,
    qualification,
    completion_year,
    unemployed_days,
    employment_status,
    studying_higher,
    paid_apprenticeship,
    self_employed_with_loan,
    registered,
    declaration
):
    if scheme == "Choose a scheme":
        return "## Please select a scheme first."

    if scheme == "Yuva Nidhi (historical reference only)":
        results = evaluate_yuva_nidhi(
            state_connection,
            qualification,
            completion_year,
            unemployed_days,
            employment_status,
            studying_higher,
            paid_apprenticeship,
            self_employed_with_loan,
            registered,
            declaration
        )
    else:
        return (
            "## No configured eligibility rules\n"
            "Janmitra AI does not yet have a verified rule set for this "
            "scheme. Open the official scheme directory to investigate it."
        )

    matched = sum(r["status"] == "MATCHED" for r in results)
    not_matched = sum(r["status"] == "NOT MATCHED" for r in results)
    unknown = sum(r["status"] == "UNKNOWN" for r in results)

    if not_matched > 0:
        overall = "Some historical conditions are not met"
    elif unknown > 0:
        overall = "More information is needed"
    else:
        overall = "All configured historical conditions matched"

    lines = [
        "# Janmitra AI — Eligibility Assessment",
        "",
        f"**Scheme:** {scheme}",
        f"**Assessment generated:** {datetime.now().strftime('%d %b %Y, %H:%M')}",
        "",
        "## Result",
        f"**{overall}**",
        "",
        f"- Matched conditions: **{matched}**",
        f"- Not matched conditions: **{not_matched}**",
        f"- Unknown conditions: **{unknown}**",
        "",
        "> **Important:** This scheme uses a historical 2022-23 "
        "Yuva Nidhi criteria snapshot from a government research report. "
        "It does not confirm eligibility under the current 2026 rules. "
        "A NOT MATCHED result applies only to this configured snapshot. "
        "Only the responsible government authority can determine official eligibility.",
        "",
        "## Condition-by-condition results",
        ""
    ]

    labels = {
        "MATCHED": "MATCHED",
        "NOT MATCHED": "NOT MATCHED",
        "UNKNOWN": "UNKNOWN"
    }

    for item in results:
        lines.extend([
            f"### {labels[item['status']]} — {item['condition']}",
            item["explanation"],
            ""
        ])

    lines.extend([
        "## Official verification",
        "",
        f"- [Seva One — Government of Karnataka]({SEVA_ONE})",
        f"- [Official Know Your Eligibility checker]({ELIGIBILITY_URL})",
        f"- [Karnataka schemes on myScheme]({MYSCHEME_KARNATAKA})",
        f"- [Government research report describing historical criteria]({YUVANIDHI_REPORT})",
        "",
        "## Next steps",
        "",
        "1. Open the official eligibility checker.",
        "2. Find the current Yuva Nidhi scheme information and confirm "
        "the applicable graduate/diploma cohort.",
        "3. Confirm all eligibility and exclusion rules with the current "
        "official scheme instructions.",
        "4. Apply only through the official application channel if it is "
        "currently open and you meet the published conditions.",
        "",
        "**Privacy:** This prototype does not store form responses in a "
        "database. Do not enter Aadhaar numbers, OTPs, passwords, or bank details."
    ])

    return "\n".join(lines)


# ------------------------------------------------------------
# GRADIO INTERFACE
# ------------------------------------------------------------

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Janmitra AI - Eligibility Assistant"
) as demo:

    gr.Markdown("""
    # 🏛️ Janmitra AI
    ### Karnataka Government Scheme Eligibility Assistant

    Evaluate applicant answers against a configured rule set.
    Every condition is marked **MATCHED**, **NOT MATCHED**, or **UNKNOWN**.

    This is an independent assistance prototype, not a government portal.
    """)

    scheme = gr.Dropdown(
        choices=[
            "Choose a scheme",
            "Yuva Nidhi (historical reference only)"
        ],
        value="Choose a scheme",
        label="Select scheme"
    )

    gr.Markdown("""
    ### Applicant information

    The current Yuva Nidhi rule set is historical. It refers to a
    2022-23 cohort described in a government research report.
    It must not be treated as current eligibility guidance.
    """)

    with gr.Row():
        with gr.Column():
            state_connection = gr.Radio(
                ["Yes", "No", "Not sure"],
                value="Not sure",
                label=(
                    "Do you meet the six-year Karnataka state connection "
                    "described in the historical reference?"
                )
            )

            qualification = gr.Radio(
                ["Degree", "Diploma", "Other", "Not sure"],
                value="Not sure",
                label="Completed qualification"
            )

            completion_year = gr.Radio(
                ["2022-23", "Other year", "Not sure"],
                value="Not sure",
                label="Academic year of course completion"
            )

            unemployed_days = gr.Number(
                value=None,
                minimum=0,
                maximum=5000,
                label="Days unemployed after course completion"
            )

            employment_status = gr.Radio(
                ["Unemployed", "Employed", "Self-employed", "Not sure"],
                value="Not sure",
                label="Current employment status"
            )

        with gr.Column():
            studying_higher = gr.Radio(
                ["Yes", "No", "Not sure"],
                value="Not sure",
                label="Are you currently pursuing higher education?"
            )

            paid_apprenticeship = gr.Radio(
                ["Yes", "No", "Not sure"],
                value="Not sure",
                label="Are you receiving wages through an apprenticeship?"
            )

            self_employed_with_loan = gr.Radio(
                ["Yes", "No", "Not sure"],
                value="Not sure",
                label=(
                    "Are you self-employed with a relevant bank/state/"
                    "central government scheme loan?"
                )
            )

            registered = gr.Radio(
                ["Yes", "No", "Not sure"],
                value="Not sure",
                label="Have you registered on the prescribed government portal?"
            )

            declaration = gr.Radio(
                ["Yes", "No", "Not sure"],
                value="Not sure",
                label="Have you submitted the required unemployment declaration?"
            )

    with gr.Row():
        assess_button = gr.Button(
            "Assess Conditions",
            variant="primary"
        )
        clear_button = gr.Button("Clear Form")

    output = gr.Markdown()

    inputs = [
        scheme,
        state_connection,
        qualification,
        completion_year,
        unemployed_days,
        employment_status,
        studying_higher,
        paid_apprenticeship,
        self_employed_with_loan,
        registered,
        declaration
    ]

    assess_button.click(
        fn=build_report,
        inputs=inputs,
        outputs=output
    )

    clear_button.click(
        fn=lambda: (
            "Choose a scheme",
            "Not sure",
            "Not sure",
            "Not sure",
            None,
            "Not sure",
            "Not sure",
            "Not sure",
            "Not sure",
            "Not sure",
            "Not sure",
            ""
        ),
        inputs=[],
        outputs=inputs + [output]
    )

    gr.Markdown("""
    ---
    ### Official resources

    - [Seva One — Karnataka Government](https://sevaone.karnataka.gov.in/)
    - [Know Your Eligibility](https://sevaone.karnataka.gov.in/services/knowYourEligibility.html)
    - [Karnataka schemes on myScheme](https://www.myscheme.gov.in/search/state/Karnataka)

    **Current limitation:** Only the historical Yuva Nidhi example has
    configured rules. Other schemes must be added after verifying their
    current official eligibility criteria.
    """)

demo.launch(share=True, debug=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.8 MB/s eta 0:00:00


/tmp/ipykernel_943/3545609549.py:376: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://833b6f7694f0a60fb8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
